In [19]:
from collections.abc import Callable

import pandas as pd
from lang_complexity import complexities

from curriculum_learning.data import heuristic_functions, load_base_dataset


def calculate_difficulty(
    dataframe: pd.DataFrame,
    heuristic_fn: Callable = heuristic_functions["word_count"],
    metric_name: str = "pragmatic_deletion_bzip2",
) -> pd.Series:
    df = dataframe
    group_size = 250

    if metric_name not in complexities:
        raise ValueError(f"Metric '{metric_name}' is not available in complexities.")
    metric_fn = complexities[metric_name].compute

    if df.empty:
        return pd.Series(dtype=float, name="difficulty")

    heuristic_score = df.apply(heuristic_fn, axis=1)
    sorted_index = heuristic_score.sort_values().index

    difficulty = pd.Series(index=df.index, dtype=float, name="difficulty")

    for start in range(0, len(sorted_index), group_size):
        group_index = sorted_index[start : start + group_size]
        group = df.loc[group_index]
        text = "\n".join(group["premise"].tolist() + group["hypothesis"].tolist())
        difficulty.loc[group_index] = metric_fn(text)

    return difficulty

base_dataset = load_base_dataset()
train_dataset = base_dataset["train"]



0       0.952592
1       0.958094
2       0.952429
3       0.941430
4       0.941580
          ...   
6495    0.949763
6496    0.948254
6497    0.953083
6498    0.949763
6499    0.951493
Name: difficulty, Length: 6500, dtype: float64

In [22]:
pd.qcut([1,2,3], 3)

[(0.999, 1.667], (1.667, 2.333], (2.333, 3.0]]
Categories (3, interval[float64, right]): [(0.999, 1.667] < (1.667, 2.333] < (2.333, 3.0]]

In [ ]:
import wandb
from curriculum_learning.config import WANDB_ORG_NAME, WANDB_PROJECT_NAME

api = wandb.Api()

runs = api.runs(f"{WANDB_ORG_NAME}/{WANDB_PROJECT_NAME}")


/home/pollux/workspace/curriculum-learning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
